In [ ]:

# BIA RAW IMPORT -> PhA_50k -> PRE/POST WINDOWS -> OUTPUT VARS
# RAW, NO FILTERING / NO RESAMPLING IMPOSED

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.stats.diagnostic import acorr_ljungbox


# CONFIG 


BIA_PKL_PATH = "LEA_BIA_RAW.pkl"  # pkl file exported from BIA device (raw data for pandas)
# BIA_PKL_PATH = "../data/LEA_BIA_RAW.pkl"   # pkl file exported from BIA device (raw data for pandas)
FREQ_COL = "f_48800"              # ~50 kHz complex impedance column

# Window timestamps (BIA datetime): 3 minutes BEFORE / fatigue inducing protocole / and 3 minutes AFTER
PRE_START_TIME  = "2025-11-28 14:57:02.563"
PRE_END_TIME    = "2025-11-28 15:00:02.563"
POST_START_TIME = "2025-11-28 15:05:45.278"
POST_END_TIME   = "2025-11-28 15:08:44.534"


# LOAD RAW BIA (.pkl)
data_bia_raw = pd.read_pickle(BIA_PKL_PATH)


# ANALYSIS DATAFRAME (same data, cleaner columns)
data_bia = data_bia_raw.copy()

# Parse time
data_bia["time"] = pd.to_datetime(data_bia["timestamp"], errors="coerce")
data_bia = data_bia.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

# Complex impedance at ~50 kHz
data_bia["Z_50k"] = data_bia[FREQ_COL].astype(np.complex128)

# Compute R, Xc, PhA (standard convention: Xc = -imag(Z))
data_bia["R_50k_ohm"]   = np.real(data_bia["Z_50k"])
data_bia["Xc_50k_ohm"]  = -np.imag(data_bia["Z_50k"])
data_bia["PhA_50k_deg"] = np.degrees(np.arctan2(data_bia["Xc_50k_ohm"], data_bia["R_50k_ohm"]))

# Compact analysis view
analysis_cols = ["time", "Z_50k", "R_50k_ohm", "Xc_50k_ohm", "PhA_50k_deg", "sat", "min", "max"]
data_bia_analysis = data_bia[analysis_cols].copy()

# CREATE WINDOWS (PRE / POST)

pre_start  = pd.to_datetime(PRE_START_TIME)
pre_end    = pd.to_datetime(PRE_END_TIME)
post_start = pd.to_datetime(POST_START_TIME)
post_end   = pd.to_datetime(POST_END_TIME)

def slice_window(df, t0, t1):
    m = (df["time"] >= t0) & (df["time"] <= t1)  # inclusive bounds [start, end]
    return df.loc[m].copy()

bia_pre  = slice_window(data_bia_analysis, pre_start, pre_end)
bia_post = slice_window(data_bia_analysis, post_start, post_end)


#  PLOTS (just for verification)

def plot_window(df_win, title, y_col="PhA_50k_deg", smooth_n=9):
    if len(df_win) < 5:
        print("Not enough points to plot:", title)
        return
    d = df_win.copy()
    d["time"] = pd.to_datetime(d["time"])
    d = d.sort_values("time")
    y = pd.to_numeric(d[y_col], errors="coerce")
    y_sm = y.rolling(smooth_n, center=True, min_periods=1).mean()

    plt.figure(figsize=(10,3))
    plt.plot(d["time"], y, alpha=0.35, label="raw")
    plt.plot(d["time"], y_sm, label=f"rolling mean (n={smooth_n})")
    plt.title(title)
    plt.xlabel("time")
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.legend()
    plt.show()

plot_window(bia_pre,  "BIA PRE (PhA_50k_deg)")
plot_window(bia_post, "BIA POST (PhA_50k_deg)")


# OUTPUT (variables to use)

pha_pre  = bia_pre["PhA_50k_deg"].astype(float).dropna().to_numpy()
pha_post = bia_post["PhA_50k_deg"].astype(float).dropna().to_numpy()

t_pre  = (bia_pre["time"]  - bia_pre["time"].iloc[0]).dt.total_seconds().to_numpy()
t_post = (bia_post["time"] - bia_post["time"].iloc[0]).dt.total_seconds().to_numpy()

dt_pre_med  = float(bia_pre["time"].diff().dt.total_seconds().median())
dt_post_med = float(bia_post["time"].diff().dt.total_seconds().median())
fs_pre_est  = 1.0 / dt_pre_med
fs_post_est = 1.0 / dt_post_med

print("\nVARIABLES À UTILISER")
print("- bia_pre   : DataFrame PRE (colonnes: time, PhA_50k_deg, etc.)")
print("- bia_post  : DataFrame POST (colonnes: time, PhA_50k_deg, etc.)")
print("- pha_pre   : numpy array, PhA_50k_deg sur PRE")
print("- pha_post  : numpy array, PhA_50k_deg sur POST")
print("- t_pre     : numpy array, temps (s) relatif au début de PRE")
print("- t_post    : numpy array, temps (s) relatif au début de POST")
print("- fs_pre_est  : float, fs approx PRE (= 1 / dt_médian)")
print("- fs_post_est : float, fs approx POST (= 1 / dt_médian)")



In [ ]:

# Global config 
# ----------

Y_COL = "PhA_50k_deg"

# We regularize the time grid because ACF/AR assume equally spaced samples.
# 500 ms is close to the observed median dt (~0.513 s) in your windows.
RESAMPLE_RULE = "500ms"

# Lags used for ACF/PACF plots:
# 40 lags at 0.5 s -> ~20 s horizon (enough to see medium-term dependence).
PLOT_LAGS = 40

# Residual whiteness test horizon:
# 20 lags at 0.5 s -> ~10 s horizon (short-to-medium memory check).
LB_LAG = 20

# Candidate AR orders upper bound (clipped further by n//10 below).
P_MAX_CAP = 20

# Significance level for residual whiteness decision.
ALPHA = 0.05


In [ ]:
# regularizes the sampling grid to satisfy the assumptions of ACF and AR models, while keeping the resampling interval close to the observed median sampling period.

def prep_series(df_win, y_col=Y_COL, resample_rule=RESAMPLE_RULE):
    """
    Prepare a clean 1D series for time-series tools (ACF/AR):
      - parse/sort timestamps
      - keep the signal as numeric
      - regularize sampling (resample + interpolate) to enforce equal spacing

    Returns:
      y_out : pd.Series indexed by time
      info  : small dict (n, dt medians, start/end) for traceability
    """
    d = df_win[["time", y_col]].copy()
    d["time"] = pd.to_datetime(d["time"], errors="coerce")
    d = d.dropna(subset=["time"]).sort_values("time").set_index("time")

    y = pd.to_numeric(d[y_col], errors="coerce")

    # Raw dt (before resampling)
    dt_raw = d.index.to_series().diff().dt.total_seconds().dropna()
    dt_raw_med = float(dt_raw.median()) if len(dt_raw) else np.nan

    if resample_rule is None:
        # If raw dt is already regular enough, keep raw sampling
        y_out = y.dropna()
        dt_out = y_out.index.to_series().diff().dt.total_seconds().dropna()
    else:
        # Regular grid -> helps make ACF/AR assumptions explicit
        y_out = y.resample(resample_rule).mean().interpolate("time")
        dt_out = y_out.index.to_series().diff().dt.total_seconds().dropna()

    info = {
        "start": d.index.min(),
        "end": d.index.max(),
        "n_raw": int(y.notna().sum()),
        "n_out": int(y_out.notna().sum()),
        "dt_raw_med_s": dt_raw_med,
        "dt_out_med_s": float(dt_out.median()) if len(dt_out) else np.nan,
        "resample_rule": resample_rule,
    }
    return y_out, info


# Prepare both windows (this is the input for all later steps)
y_pre, info_pre = prep_series(bia_pre)
y_post, info_post = prep_series(bia_post)

print("PRE :", f"n_raw={info_pre['n_raw']}, n_out={info_pre['n_out']}, "
             f"dt_raw_med={info_pre['dt_raw_med_s']:.3f}s, dt_out_med={info_pre['dt_out_med_s']:.3f}s, "
             f"range={info_pre['start']} -> {info_pre['end']}, resample={info_pre['resample_rule']}")

print("POST:", f"n_raw={info_post['n_raw']}, n_out={info_post['n_out']}, "
             f"dt_raw_med={info_post['dt_raw_med_s']:.3f}s, dt_out_med={info_post['dt_out_med_s']:.3f}s, "
             f"range={info_post['start']} -> {info_post['end']}, resample={info_post['resample_rule']}")


In [ ]:
# general plot, just for visual inspection.
# note : Lag 0 was excluded from ACF and PACF plots since it is trivially equal to 1 adds nothing for model identification (also it clipps into into the plot area).


from statsmodels.tsa.stattools import acf, pacf

def plot_series_acf_pacf(y, name, plot_lags=PLOT_LAGS):
    # 1) Series
    plt.figure(figsize=(10, 3))
    plt.plot(y.index, y.values)
    plt.title(f"{name} | {Y_COL} (prepared)")
    plt.xlabel("time")
    plt.ylabel(Y_COL)
    plt.tight_layout()
    plt.show()

    # Compute ACF / PACF explicitly
    acf_vals = acf(y.dropna(), nlags=plot_lags, fft=True)
    pacf_vals = pacf(y.dropna(), nlags=plot_lags, method="ywm")

    lags = np.arange(1, plot_lags + 1)

    # 2) ACF (exclude lag 0)
    plt.figure(figsize=(8, 3))
    plt.stem(lags, acf_vals[1:], basefmt=" ")
    plt.axhline(0, color="k", linewidth=0.8)
    plt.axhspan(
        -1.96 / np.sqrt(len(y)),
         1.96 / np.sqrt(len(y)),
        alpha=0.2
    )
    plt.title(f"{name} | ACF (lags 1–{plot_lags})")
    plt.xlabel("lag")
    plt.tight_layout()
    plt.show()

    # 3) PACF (exclude lag 0)
    plt.figure(figsize=(8, 3))
    plt.stem(lags, pacf_vals[1:], basefmt=" ")
    plt.axhline(0, color="k", linewidth=0.8)
    plt.axhspan(
        -1.96 / np.sqrt(len(y)),
         1.96 / np.sqrt(len(y)),
        alpha=0.2
    )
    plt.title(f"{name} | PACF (lags 1–{plot_lags})")
    plt.xlabel("lag")
    plt.tight_layout()
    plt.show()



# Plots for narrative: first PRE, then POST
plot_series_acf_pacf(y_pre, "PRE")
plot_series_acf_pacf(y_post, "POST")


In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

warnings.filterwarnings("ignore", category=InterpolationWarning)

# ^^^ fix this later lol. still valid but some results are spamming warnings dues to being so far under 0.01 p-values. (KPSS test)

def stationarity_tests(y):
    """
    Two complementary tests:
      - ADF: H0 = unit root (non-stationary). Small p -> reject non-stationarity.
      - KPSS: H0 = stationary. Small p -> reject stationarity.

    Using both helps avoid a one-test-only decision.
    """
    yv = pd.Series(y).dropna().to_numpy()

    # ADF (autolag chooses lag length via AIC internally)
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(yv, autolag="AIC")

    # KPSS can warn when p-value is below available table -> interpret as p < 0.01
    try:
        kpss_stat, kpss_p, _, kpss_crit = kpss(yv, regression="c", nlags="auto")
    except Exception:
        kpss_stat, kpss_p, kpss_crit = np.nan, np.nan, None

    return {
        "adf_stat": float(adf_stat),
        "adf_p": float(adf_p),
        "adf_crit": adf_crit,
        "kpss_stat": float(kpss_stat) if np.isfinite(kpss_stat) else np.nan,
        "kpss_p": float(kpss_p) if np.isfinite(kpss_p) else np.nan,
        "kpss_crit": kpss_crit,
    }


def print_stationarity(st, name):
    print(f"\n{name} | stationarity tests")
    print(f"ADF : stat={st['adf_stat']:.3f} | p={st['adf_p']:.4f}")
    if np.isfinite(st["kpss_stat"]):
        if st["kpss_p"] <= 0.01:
            print(f"KPSS: stat={st['kpss_stat']:.3f} | p<0.01")
        else:
            print(f"KPSS: stat={st['kpss_stat']:.3f} | p={st['kpss_p']:.4f}")
    else:
        print("KPSS: NA (series too short or nearly constant)")


# Run tests on prepared series 
st_pre = stationarity_tests(y_pre)
st_post = stationarity_tests(y_post)

print_stationarity(st_pre, "PRE")
print_stationarity(st_post, "POST")


# Decision step for AR modeling 
# AR models assume stationarity; if ADF does not reject non-stationarity (p>0.05),
# we use first difference (diff1) as a simple stationarization step.
y_pre_fit = y_pre.dropna()
y_post_fit = y_post.dropna()

diff_pre = False
diff_post = False

if st_pre["adf_p"] > 0.05:
    y_pre_fit = y_pre_fit.diff().dropna()
    diff_pre = True

if st_post["adf_p"] > 0.05:
    y_post_fit = y_post_fit.diff().dropna()
    diff_post = True

print("\nModeling series (after stationarity step)")
print(f"PRE : diff1={diff_pre} | n={len(y_pre_fit)}")
print(f"POST: diff1={diff_post} | n={len(y_post_fit)}")


In [ ]:
import warnings

# Optional: keep the notebook clean (does not change results)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

def ar_grid_search(y_fit, p_max_cap=P_MAX_CAP):
    """
    Fit AR(p) for p=1..p_max and report best AIC and best BIC
    cap p to avoid overfitting on short windows
    """
    yv = pd.Series(y_fit).dropna()
    n = len(yv)
    p_max = max(1, min(p_max_cap, n // 10))  # conservative rule

    best_aic = None
    best_bic = None

    for p in range(1, p_max + 1):
        try:
            m = AutoReg(yv, lags=p, old_names=False).fit()
        except Exception:
            continue

        if (best_aic is None) or (m.aic < best_aic.aic):
            best_aic = m
        if (best_bic is None) or (m.bic < best_bic.bic):
            best_bic = m

    if best_aic is None or best_bic is None:
        raise RuntimeError("AR grid search failed (try smaller p_max_cap).")

    return {
        "p_max_used": p_max,
        "model_aic": best_aic,
        "p_aic": int(best_aic.model._maxlag),
        "model_bic": best_bic,
        "p_bic": int(best_bic.model._maxlag),
    }


def choose_p_by_whiteness(y_fit, p_max_cap=P_MAX_CAP, lb_lag=LB_LAG, alpha=ALPHA):
    """
    choose the smallest p such that Ljung-Box p-value > alpha.
    If none passes, fall back to best-BIC (+report the limitation).
    """
    yv = pd.Series(y_fit).dropna()
    n = len(yv)
    p_max = max(1, min(p_max_cap, n // 10))

    rows = []
    models = {}

    for p in range(1, p_max + 1):
        try:
            m = AutoReg(yv, lags=p, old_names=False).fit()
            resid = pd.Series(m.resid).dropna()

            lb = acorr_ljungbox(resid, lags=[lb_lag], return_df=True)
            lb_p = float(lb["lb_pvalue"].iloc[0])

            rows.append((p, float(m.aic), float(m.bic), lb_p))
            models[p] = m
        except Exception:
            continue

    scan = pd.DataFrame(rows, columns=["p", "AIC", "BIC", f"LB_p_lag{lb_lag}"]).sort_values("p")

    ok = scan[scan[f"LB_p_lag{lb_lag}"] > alpha]
    if len(ok):
        p_star = int(ok.iloc[0]["p"])
        return {"p": p_star, "model": models[p_star], "passed": True, "p_max_used": p_max}
    else:
        # fallback: best BIC among scanned models
        p_bic = int(scan.loc[scan["BIC"].idxmin(), "p"])
        return {"p": p_bic, "model": models[p_bic], "passed": False, "p_max_used": p_max}


from statsmodels.tsa.stattools import acf

def residual_diagnostics(model, name, lb_lag=LB_LAG, plot_lags=PLOT_LAGS):
    """
    Minimal diagnostics:
      - Residual ACF (visual) excluding lag 0 (trivially = 1)
      - Ljung-Box p-value at fixed lag (statistical)
    """
    resid = pd.Series(model.resid).dropna()

    # Compute residual ACF explicitly to drop lag 0 from the plot. see before notes (it does skew the view here as well, since the scale is dominated by lag 0).
    acf_vals = acf(resid.to_numpy(), nlags=plot_lags, fft=True)
    lags = np.arange(1, plot_lags + 1)

    plt.figure(figsize=(8, 3))
    plt.stem(lags, acf_vals[1:], basefmt=" ")
    plt.axhline(0, color="k", linewidth=0.8)

    # simple approx CI band (same idea as statsmodels' default)
    ci = 1.96 / np.sqrt(len(resid))
    plt.axhspan(-ci, ci, alpha=0.2)

    plt.title(f"{name} | Residual ACF (lags 1–{plot_lags})")
    plt.xlabel("lag")
    plt.tight_layout()
    plt.show()

    # Ljung-Box whiteness test at a fixed horizon
    lb = acorr_ljungbox(resid, lags=[lb_lag], return_df=True)
    lb_p = float(lb["lb_pvalue"].iloc[0])

    return {
        "lb_p": lb_p,
        "resid_var": float(np.var(resid, ddof=1)),
    }



def run_ar_block(y_fit, name, diff_used):
    """
    One window (PRE or POST):
      - report best AIC and best BIC AR(p)
      - pick final p by whiteness 
      - report a small set of parameters
      - residual diagnostics
    """
    print(f"\n {name} | AR modeling")
    print(f"Series used: diff1={diff_used} | n={len(y_fit)}")

    grid = ar_grid_search(y_fit)
    print(f"AR grid search: p_max_used={grid['p_max_used']}")
    print(f"Best by AIC: p={grid['p_aic']} | AIC={grid['model_aic'].aic:.3f} | BIC={grid['model_aic'].bic:.3f}")
    print(f"Best by BIC: p={grid['p_bic']} | AIC={grid['model_bic'].aic:.3f} | BIC={grid['model_bic'].bic:.3f}")

    chosen = choose_p_by_whiteness(y_fit)
    final_p = chosen["p"]
    final_model = chosen["model"]

    if chosen["passed"]:
        print(f"Final AR(p) chosen by whiteness: p={final_p}")
    else:
        print(f"Final AR(p): no p<=p_max achieved Ljung-Box p>{ALPHA} at lag={LB_LAG}.")
        print(f"Using best-BIC fallback: p={final_p}")
        print("Note: remaining autocorrelation suggests ARMA may be more appropriate.")

    # Minimal parameter report (enough for the assignment)
    print("Params (head):")
    print(final_model.params.head(6))

    diag = residual_diagnostics(final_model, name)

    print(f"Ljung-Box (lag={LB_LAG}): p={diag['lb_p']:.4f}")

    return {
        "name": name,
        "diff1": diff_used,
        "p_aic": grid["p_aic"],
        "p_bic": grid["p_bic"],
        "p_final": int(final_p),
        "aic_final": float(final_model.aic),
        "bic_final": float(final_model.bic),
        "lb_p": diag["lb_p"],
        "resid_var": diag["resid_var"],
    }


# Run both windows
out_pre = run_ar_block(y_pre_fit, "PRE", diff_pre)
out_post = run_ar_block(y_post_fit, "POST", diff_post)


In [ ]:
summary = pd.DataFrame([
    {
        "Window": "PRE",
        "n_out": info_pre["n_out"],
        "dt_out_med_s": info_pre["dt_out_med_s"],
        "ADF_p": st_pre["adf_p"],
        "KPSS_p": st_pre["kpss_p"],   # may be <=0.01
        "diff1": diff_pre,
        "p_AIC": out_pre["p_aic"],
        "p_BIC": out_pre["p_bic"],
        "p_final": out_pre["p_final"],
        "AIC_final": out_pre["aic_final"],
        "BIC_final": out_pre["bic_final"],
        "LB_p_lag20": out_pre["lb_p"],
        "resid_var": out_pre["resid_var"],
    },
    {
        "Window": "POST",
        "n_out": info_post["n_out"],
        "dt_out_med_s": info_post["dt_out_med_s"],
        "ADF_p": st_post["adf_p"],
        "KPSS_p": st_post["kpss_p"],
        "diff1": diff_post,
        "p_AIC": out_post["p_aic"],
        "p_BIC": out_post["p_bic"],
        "p_final": out_post["p_final"],
        "AIC_final": out_post["aic_final"],
        "BIC_final": out_post["bic_final"],
        "LB_p_lag20": out_post["lb_p"],
        "resid_var": out_post["resid_var"],
    },
])

# Compact display (rounded)
summary_print = summary.copy()
for c in ["dt_out_med_s","ADF_p","KPSS_p","AIC_final","BIC_final","LB_p_lag20","resid_var"]:
    summary_print[c] = summary_print[c].astype(float).round(6)

summary_print["KPSS_p"] = summary_print["KPSS_p"].apply(lambda v: "<0.01" if float(v) <= 0.01 else f"{float(v):.4f}") # interpret small p per earlier note

print("\nPRE vs POST (summary)")
print(summary_print.to_string(index=False))
